# Experimentul 3: VQC cu ZZ feature map redus (2 → 1 repetări)

**O singură modificare față de baseline: `FEATURE_MAP_REPS = 1`. Antrenare fără zgomot, evaluare pe validation la p = 0, 0.001, 0.01, 0.05.**

## Motivație (din Experimentul 2)
În baseline, feature map-ul are 52 din cele 93 de porți native, 24 din cele 33 de CNOT-uri și 76 din cele 126 de locuri de zgomot. Pe validation, robustețea cedează între p = 0.01 și p = 0.05 (Spearman ρ 0.982 → 0.684, AUPRC 0.546 → 0.381).

## Ipoteze (de testat, nu presupuse)
- **H1:** cu o singură repetare (67 de porți, 21 CNOT, 88 de locuri de zgomot) circuitul acumulează mai puțin zgomot, deci AUPRC și ρ la p = 0.01 și 0.05 scad mai puțin decât la baseline.
- **H0 / alternativă:** o singură repetare encodează datele mai slab (mai puține produse între features), deci modelul pierde la p = 0 mai mult decât câștigă în robustețe; sau zgomotul din ansatz domină oricum.

## Ce ținem fix (identic cu Experimentele 1 și 2)
Splitul comun (`split_indices.npz`), raportul 1:1 = `undersample(X_train, y_train, seed)`, `AngleEncoder(4, seed)` fitat pe setul 1:1, 4 qubiți, ansatz EfficientSU2-style cu 3 straturi (32 de parametri), readout pe qubitul 0, Adam, lr 0.05, batch 32, 30 de epoci, epoca aleasă după AUPRC pe validation, seeds `[0, 1, 2, 3, 4]`, modelul de zgomot oficial (`noisy_qnode`), pragul max-F1 pe validation la același p.

**Comparație pereche.** Pentru un seed dat, Exp. 2 și Exp. 3 au același set de antrenare, același encoder și **aceleași ponderi inițiale** (`init_weights(seed)` nu depinde de feature map). Singura diferență între perechi este circuitul, așa că diferențele se citesc seed cu seed.

**Testul nu se folosește** (`RUN_TEST = False`): matricea lui de features este ștearsă din memorie.

## 1. Configurație și imports

**Seeds.** `set_seed` fixează numpy și torch; amestecarea folosește un `torch.Generator` dedicat (ca în Experimentele 1 și 2). Simulatoarele PennyLane rulează cu `shots=None`; singura excepție este numărarea execuțiilor parameter-shift cu `qml.Tracker` (secțiunea 3), pe un dispozitiv cu shots și `seed` fix.

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# ---- etape ----
RUN_TEST = False              # False: doar validation. Testul se folosește doar la evaluarea finală.

# ---- singura modificare față de baseline ----
FEATURE_MAP_REPS = 1          # baseline (Exp. 2): 2
BASELINE_FEATURE_MAP_REPS = 2 # doar pentru comparația structurală cu baseline-ul

# ---- fix, identic cu Experimentele 1 și 2 ----
SPLIT_SEED = 42
N_QUBITS = 4
ANSATZ_REPS = 3               # (3 + 1) * 4 * 2 = 32 parametri
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.05
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger
NOISE_LEVELS = [0.001, 0.01, 0.05]
NATIVE_GATES = {"RX", "RY", "RZ", "Rot", "Hadamard", "CNOT", "CZ"}
SEEDS = [0, 1, 2, 3, 4]
P_ALL = [0.0, *NOISE_LEVELS]

EXPERIMENT_NAME = "reduced_feature_map"
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
RESULTS_DIR = REPO_ROOT / "results" / "result_exp3"
WEIGHTS_DIR = RESULTS_DIR / "weights"
EXP2_DIR = REPO_ROOT / "results" / "result_exp2"                                # baseline: doar citit
EXP1_DIR = REPO_ROOT / "results" / "result_exp1" / "test1_full_without_noise"   # doar citit (costul antrenării baseline)
SPLIT_PATH = REPO_ROOT / "split_indices.npz"

BASE_COLOR, NEW_COLOR, SEED_DOT_COLOR = "#2a78d6", "#eb6834", "#8a8f98"
BASE_LABEL, NEW_LABEL = "baseline Exp. 2 (ZZ ×2)", "Exp. 3 (ZZ ×1)"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SPLIT_SEED)
print(f"FEATURE_MAP_REPS = {FEATURE_MAP_REPS} (baseline {BASELINE_FEATURE_MAP_REPS})   seeds: {SEEDS}   p: {P_ALL}   RUN_TEST: {RUN_TEST}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__)
print("rezultate:", RESULTS_DIR.relative_to(REPO_ROOT))

## 2. Datele și splitul comun

Splitul se încarcă din `split_indices.npz` (obligatoriu), în ordinea salvată: ordinea rândurilor din training contează pentru `undersample`. Verificăm că fișierul coincide cu reconstrucția oficială (Partea 4) și cu numărul de fraude 295 / 99 / 98.

In [ ]:
DATA_PATH = REPO_ROOT / "data" / "creditcard.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"fișier obligatoriu lipsă: {DATA_PATH}")
if not SPLIT_PATH.exists():
    raise FileNotFoundError(f"fișier obligatoriu lipsă: {SPLIT_PATH}")
df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and "Time" in df.columns and df.index.equals(pd.RangeIndex(len(df)))

X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

# verificare: fișierul coincide cu splitul oficial reconstruit
_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx), "split_indices.npz diferă de splitul oficial"
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
y_test = y_all[test_idx]
n_test_rows, n_test_frauds = len(test_idx), int(y_test.sum())
assert (int(y_train.sum()), int(y_val.sum()), n_test_frauds) == (295, 99, 98) and n_test_rows == 56962
if RUN_TEST:
    X_test = X_all.loc[test_idx]
del X_all, df, _trval, _test, _train, _val   # etapa pe validation: features-urile testului nu rămân în memorie

print(f"split: {SPLIT_PATH.name} (identic cu reconstrucția oficială)")
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())})   test {n_test_rows:,} ({n_test_frauds}, doar etichete)")

## 3. Funcțiile din notebookul oficial și structura circuitului

Funcțiile sunt cele din notebookul oficial (golurile completate), identice cu cele din Experimentul 2; `feature_map` citește `FEATURE_MAP_REPS`, deci aici construiește o singură repetare. `train_vqc_run` este bucla din Experimentul 1.

In [ ]:
def undersample(X_part, y_part, seed):
    rng = np.random.default_rng(seed)
    fraud = np.flatnonzero(y_part == 1)
    legit = rng.choice(np.flatnonzero(y_part == 0), size=len(fraud), replace=False)
    keep = rng.permutation(np.concatenate([fraud, legit]))
    return X_part.iloc[keep], y_part[keep]


class AngleEncoder:
    raw_columns = ["Time", "Amount"]

    def __init__(self, n_components, seed):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=n_components, random_state=seed)
        self.angles = MinMaxScaler(feature_range=(0, np.pi))

    def _scaled(self, frame):
        out = frame.copy()
        out[self.raw_columns] = self.scaler.transform(frame[self.raw_columns])
        return out.to_numpy()

    def fit(self, frame):
        self.scaler.fit(frame[self.raw_columns])
        self.angles.fit(self.pca.fit_transform(self._scaled(frame)))
        return self

    def transform(self, frame):
        return np.clip(self.angles.transform(self.pca.transform(self._scaled(frame))), 0, np.pi)


def feature_map(x):
    """ZZ feature map. x holds N_QUBITS angles in [0, pi], or a batch of them (shape (..., N_QUBITS))."""
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[..., i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * (np.pi - x[..., i]) * (np.pi - x[..., j]), wires=j)
                qml.CNOT(wires=[i, j])


def ansatz(weights):
    """EfficientSU2-style ansatz; weights has shape (ANSATZ_REPS + 1, N_QUBITS, 2)."""
    for layer in range(ANSATZ_REPS + 1):
        for i in range(N_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        if layer < ANSATZ_REPS:
            for i in range(N_QUBITS - 1):
                qml.CNOT(wires=[i, i + 1])


def vqc(x, weights):
    feature_map(x)
    ansatz(weights)
    return qml.expval(qml.PauliZ(0))


device = qml.device("default.qubit", wires=N_QUBITS)   # shots=None: valori de așteptare exacte
circuit = qml.QNode(vqc, device, interface="torch", diff_method="backprop")


def fraud_probability(x, weights):
    return (1 - circuit(x, weights)) / 2


def init_weights(seed):
    generator = torch.Generator().manual_seed(seed)
    return torch.nn.Parameter(2 * np.pi * torch.rand(ANSATZ_REPS + 1, N_QUBITS, 2, generator=generator, dtype=torch.float64))


def predict(Z, weights, model=None, chunk=20000):
    model = model or fraud_probability
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([model(X_t[k:k + chunk], weights) for k in range(0, len(X_t), chunk)]).numpy()


def train_vqc_run(Z_tr, y_tr, Z_valid, y_valid, initial_weights, seed, epochs=EPOCHS):
    set_seed(seed)
    shuffle_generator = torch.Generator().manual_seed(seed)
    weights = torch.nn.Parameter(initial_weights.detach().clone())   # copie independentă
    optimizer = torch.optim.Adam([weights], lr=LEARNING_RATE)
    loss_fn = torch.nn.BCELoss()
    X_t = torch.tensor(Z_tr)
    y_t = torch.tensor(y_tr, dtype=torch.float64)
    history, best_ap, best_weights, best_epoch = [], -1.0, None, None
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_t), generator=shuffle_generator)
        losses = []
        for k in range(0, len(X_t), BATCH_SIZE):
            idx = order[k:k + BATCH_SIZE]
            optimizer.zero_grad()
            loss = loss_fn(fraud_probability(X_t[idx], weights), y_t[idx])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        val_ap = average_precision_score(y_valid, predict(Z_valid, weights))
        history.append({"epoch": epoch, "train loss": float(np.mean(losses)), "validation AUPRC": float(val_ap)})
        if val_ap > best_ap:
            best_ap, best_weights, best_epoch = val_ap, weights.detach().clone(), epoch
    return {"weights": best_weights, "history": history, "best_epoch": best_epoch, "best_val_ap": float(best_ap),
            "time_train_s": time.perf_counter() - start}


def choose_threshold(y_true, scores):
    """The threshold that maximises F1 on (y_true, scores)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y_true, scores, threshold):
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "AUPRC": average_precision_score(y_true, scores),
        "AUROC": roc_auc_score(y_true, scores),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "F1": f1_score(y_true, predicted, zero_division=0),
        "MCC": matthews_corrcoef(y_true, predicted),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


def noisy_qnode(circuit_fn, p, n_wires=N_QUBITS):
    """QNode of circuit_fn under the challenge's noise model: native gates, then a depolarizing channel of
    probability p after every gate, on each of its qubits. Differentiable with backprop."""
    noisy_device = qml.noise.insert(qml.device("default.mixed", wires=n_wires), qml.DepolarizingChannel, p, position="all")
    qnode = qml.QNode(circuit_fn, noisy_device, interface="torch", diff_method="backprop")
    return qml.transforms.decompose(qnode, gate_set=NATIVE_GATES)


def noisy_predict(Z, weights, p, circuit_fn=vqc, readout=lambda expval: (1 - expval) / 2, n_wires=N_QUBITS, chunk=2000):
    """Fraud scores of the transactions Z when circuit_fn runs under depolarizing noise p."""
    noisy_circuit = noisy_qnode(circuit_fn, p, n_wires)
    X_t = torch.tensor(Z)
    with torch.no_grad():
        return torch.cat([readout(noisy_circuit(X_t[k:k + chunk], weights)) for k in range(0, len(X_t), chunk)]).numpy()


def noise_locations(circuit_fn, *args, n_wires=N_QUBITS):
    """Number of depolarizing channels the noise model inserts: one per qubit of every native gate."""
    qnode = qml.transforms.decompose(qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires)), gate_set=NATIVE_GATES)
    return sum(size * count for size, count in qml.specs(qnode)(*args)["resources"].gate_sizes.items())


def evaluate_under_noise(circuit_fn, weights, Z_valid, Z_test_part, y_valid=y_val, y_test_part=y_test,
                         readout=lambda expval: (1 - expval) / 2, noise_levels=NOISE_LEVELS, n_wires=N_QUBITS):
    clean = qml.QNode(circuit_fn, qml.device("default.qubit", wires=n_wires), interface="torch")
    with torch.no_grad():
        clean_val = readout(clean(torch.tensor(Z_valid), weights)).numpy()
        clean_test = readout(clean(torch.tensor(Z_test_part), weights)).numpy()
    rows = []
    for p in [0.0, *noise_levels]:
        if p == 0.0:
            val_scores, test_scores = clean_val, clean_test
        else:
            val_scores = noisy_predict(Z_valid, weights, p, circuit_fn, readout, n_wires)
            test_scores = noisy_predict(Z_test_part, weights, p, circuit_fn, readout, n_wires)
        threshold = choose_threshold(y_valid, val_scores)
        rows.append({"p": p, **evaluate(y_test_part, test_scores, threshold), "Spearman rho": spearmanr(clean_test, test_scores).statistic, "threshold": threshold})
    return pd.DataFrame(rows).set_index("p")

### Verificări și structura circuitului: baseline vs Exp. 3

Checks-urile din notebook (4.2, 9.1, 6.1, 6.2, 10.1) rulează pe circuitul cu o repetare. Apoi măsurăm structura **ambelor** circuite cu aceleași unelte, schimbând temporar `FEATURE_MAP_REPS`:
- porți native, CNOT, adâncime (`qml.specs` după decompoziția în `NATIVE_GATES`) și locuri de zgomot (`noise_locations`), comparate cu formula așteptată;
- execuțiile parameter-shift **măsurate** cu `qml.Tracker` pe un mini-batch de 32 (ca în Partea 8), nu presupuse. Formula nominală $1 + 2P$ depinde doar de numărul de parametri ($P = 32$ în ambele), nu de numărul de porți.

In [ ]:
X_bal_check, y_bal_check = undersample(X_train, y_train, SPLIT_SEED)   # 4.2
assert len(y_bal_check) == 2 * y_train.sum() and y_bal_check.sum() == y_train.sum() and X_bal_check.index.is_unique
assert set(X_bal_check.index) <= set(X_train.index)

rng_check = np.random.default_rng(3)   # 9.1
y_c = rng_check.integers(0, 2, 300)
s_c = np.clip(0.3 * y_c + rng_check.random(300), 0, 1)
best_f1 = max(f1_score(y_c, (s_c >= t).astype(int)) for t in np.unique(s_c))
assert np.isclose(f1_score(y_c, (s_c >= choose_threshold(y_c, s_c)).astype(int)), best_f1)

dev4 = qml.device("default.qubit", wires=N_QUBITS)


def state_of(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.QNode(full, dev4)()


def gate_counts(circuit_fn, *args):
    def full():
        circuit_fn(*args)
        return qml.state()
    return qml.specs(qml.transforms.decompose(qml.QNode(full, dev4), gate_set=NATIVE_GATES))()["resources"].gate_types


def feature_map_reference(x):
    for _ in range(FEATURE_MAP_REPS):
        for i in range(N_QUBITS):
            qml.Hadamard(wires=i)
            qml.RZ(2 * x[i], wires=i)
        for i in range(N_QUBITS):
            for j in range(i + 1, N_QUBITS):
                qml.IsingZZ(2 * (np.pi - x[i]) * (np.pi - x[j]), wires=[i, j])


Z_check = AngleEncoder(N_QUBITS, SPLIT_SEED).fit(X_bal_check).transform(X_bal_check)
for x_check in Z_check[:3]:   # 6.1, cu o repetare
    assert np.isclose(abs(np.vdot(state_of(feature_map_reference, x_check), state_of(feature_map, x_check))) ** 2, 1.0)

w_check = torch.linspace(0, 3, (ANSATZ_REPS + 1) * N_QUBITS * 2, dtype=torch.float64).reshape(ANSATZ_REPS + 1, N_QUBITS, 2)   # 6.2
probabilities = np.abs(qml.math.to_numpy(state_of(ansatz, w_check))) ** 2
assert np.allclose(probabilities[[0, 5, 10, 15]], [0.049016, 0.01134, 0.037841, 0.048166], atol=1e-5)


@qml.qnode(qml.device("default.mixed", wires=1))   # 10.1
def noisy_ry(theta, p):
    qml.RY(theta, wires=0)
    qml.DepolarizingChannel(p, wires=0)
    return qml.expval(qml.PauliZ(0))


for theta, p in [(0.3, 0.0), (1.0, 0.1), (2.5, 0.3), (0.7, 0.75)]:
    assert np.isclose(noisy_ry(theta, p), (1 - 4 * p / 3) * np.cos(theta))
print("✅ Checks 4.2, 9.1, 6.1, 6.2, 10.1: trecute (feature map cu o repetare)")


def expected_structure(reps):
    """Numărătoarea așteptată, din construcția circuitului: feature map + ansatz."""
    pairs = N_QUBITS * (N_QUBITS - 1) // 2
    fm_gates, fm_cnots = reps * (2 * N_QUBITS + 3 * pairs), reps * 2 * pairs
    an_gates, an_cnots = (ANSATZ_REPS + 1) * N_QUBITS * 2 + ANSATZ_REPS * (N_QUBITS - 1), ANSATZ_REPS * (N_QUBITS - 1)
    return {"porți native": fm_gates + an_gates, "CNOT": fm_cnots + an_cnots, "locuri de zgomot": fm_gates + an_gates + fm_cnots + an_cnots}


def circuit_structure(reps, x, batch_x, batch_y):
    """Structura și execuțiile parameter-shift ale VQC-ului cu `reps` repetări ale feature map-ului."""
    global FEATURE_MAP_REPS
    saved_reps, FEATURE_MAP_REPS = FEATURE_MAP_REPS, reps   # schimbare temporară, restaurată în finally
    try:
        w_probe = init_weights(SEEDS[0])
        counts = gate_counts(vqc, x, w_probe)
        resources = qml.specs(qml.transforms.decompose(qml.QNode(vqc, dev4), gate_set=NATIVE_GATES))(x, w_probe)["resources"]
        shot_device = qml.device("default.qubit", wires=N_QUBITS, seed=SPLIT_SEED)
        hardware_like = qml.set_shots(qml.QNode(vqc, shot_device, interface="torch", diff_method="parameter-shift"), shots=SHOTS_PER_EXECUTION)
        with qml.Tracker(shot_device) as tracker:
            loss = torch.nn.BCELoss()((1 - hardware_like(batch_x, w_probe)) / 2, batch_y)
            loss.backward()
        return {
            "repetări ZZ": reps, "porți native": sum(counts.values()), "CNOT": counts["CNOT"], "adâncime": resources.depth,
            "locuri de zgomot": noise_locations(vqc, x, w_probe), "parametri": w_probe.numel(),
            "execuții / exemplu (nominal 1 + 2P)": 1 + 2 * w_probe.numel(),
            "execuții / exemplu (Tracker, parameter shift)": tracker.totals["executions"] / len(batch_x),
        }
    finally:
        FEATURE_MAP_REPS = saved_reps


batch_x, batch_y = torch.tensor(Z_check[:BATCH_SIZE]), torch.tensor(y_bal_check[:BATCH_SIZE], dtype=torch.float64)
structure = pd.DataFrame([circuit_structure(r, torch.tensor(Z_check[0]), batch_x, batch_y)
                          for r in [BASELINE_FEATURE_MAP_REPS, FEATURE_MAP_REPS]], index=[BASE_LABEL, NEW_LABEL])
assert FEATURE_MAP_REPS == 1, "FEATURE_MAP_REPS trebuie restaurat"
for label, reps in [(BASE_LABEL, BASELINE_FEATURE_MAP_REPS), (NEW_LABEL, FEATURE_MAP_REPS)]:
    for key, value in expected_structure(reps).items():
        assert structure.loc[label, key] == value, f"{label}: {key} = {structure.loc[label, key]}, așteptat {value}"
assert tuple(structure.loc[BASE_LABEL, ["porți native", "CNOT", "locuri de zgomot"]]) == (93, 33, 126), "baseline-ul trebuie să fie cel din Exp. 2"
N_PARAMS = int(structure.loc[NEW_LABEL, "parametri"])
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
assert N_PARAMS == 32
print("✅ structură: numărătorile coincid cu formula așteptată pentru ambele circuite")
display(structure)

## 4. Antrenare fără zgomot (de la zero) și salvarea ponderilor

Pentru fiecare seed: `undersample` 1:1 → `AngleEncoder(4, seed)` → `init_weights(seed)` → `train_vqc_run`, 30 de epoci, cu epoca aleasă după AUPRC pe validation. Ponderile epocii alese și istoricul se salvează în `results/result_exp3/weights/seed_<s>.pt`, iar un rezumat per seed în `train_runs.jsonl`. La o rulare ulterioară, ponderile salvate se încarcă și se verifică (AUPRC recalculat), fără reantrenare.

Ponderile inițiale se salvează și ele și se verifică egale cu `init_weights(seed)`, aceeași funcție și același seed cu care a pornit baseline-ul: comparația pe seed este pereche.

In [ ]:
def load_jsonl(path, required=False):
    """Înregistrările din path. Un fișier obligatoriu lipsă oprește execuția, în loc să întoarcă o listă goală."""
    if not path.exists():
        if required:
            raise FileNotFoundError(f"fișier obligatoriu lipsă: {path}")
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def to_builtin(value):
    if isinstance(value, dict):
        return {k: to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_builtin(record), ensure_ascii=False) + "\n")


config = {
    "experiment": EXPERIMENT_NAME, "change_vs_baseline": "FEATURE_MAP_REPS 2 -> 1 (only change)",
    "feature_map_reps": FEATURE_MAP_REPS, "ansatz_reps": ANSATZ_REPS, "n_qubits": N_QUBITS, "n_params": N_PARAMS,
    "seeds": SEEDS, "noise_levels": P_ALL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
    "optimizer": "Adam", "loss": "BCELoss", "init": "init_weights(seed)", "shuffle": "torch.Generator().manual_seed(seed)",
    "balancing": "undersample(X_train, y_train, seed), ratio 1:1", "encoder": "AngleEncoder(4, seed) fitted on the 1:1 set",
    "epoch_selection": "best validation AUPRC", "noise_model": "noisy_qnode: NATIVE_GATES decomposition, DepolarizingChannel(p) after every gate on every wire, default.mixed",
    "threshold": "max F1 on validation scores at the same p", "evaluation_set": "validation (test not used)",
    "split": "split_indices.npz (verified against official reconstruction)", "shots_per_execution": SHOTS_PER_EXECUTION,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
}
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RESULTS_DIR / "config.json"
TRAIN_RUNS_PATH, VAL_RUNS_PATH, TEST_RUNS_PATH = RESULTS_DIR / "train_runs.jsonl", RESULTS_DIR / "val_runs.jsonl", RESULTS_DIR / "test_runs.jsonl"
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    current = json.loads(json.dumps(to_builtin(config)))
    if {k: v for k, v in previous.items() if k != "versions"} != {k: v for k, v in current.items() if k != "versions"}:
        raise RuntimeError(f"{CONFIG_PATH} conține altă configurație. Nu suprascriu: mută directorul sau schimbă RESULTS_DIR.")
else:
    CONFIG_PATH.write_text(json.dumps(to_builtin(config), indent=2, ensure_ascii=False), encoding="utf-8")

train_done = {r["seed"]: r for r in load_jsonl(TRAIN_RUNS_PATH)}
models, train_time_session = {}, 0.0
for seed in SEEDS:
    X_s, y_s = undersample(X_train, y_train, seed)
    encoder = AngleEncoder(N_QUBITS, seed).fit(X_s)
    Z_tr, Z_val = encoder.transform(X_s), encoder.transform(X_val)
    path = WEIGHTS_DIR / f"seed_{seed}.pt"
    if path.exists() and seed in train_done:
        saved = torch.load(path, weights_only=True)
        source = "încărcat"
    else:
        initial = init_weights(seed)
        assert torch.equal(initial, init_weights(seed)), "inițializarea trebuie să fie deterministă"
        run = train_vqc_run(Z_tr, y_s, Z_val, y_val, initial, seed)
        saved = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"],
                 "history": run["history"], "initial_weights": initial.detach().clone()}
        torch.save(saved, path)
        append_jsonl(TRAIN_RUNS_PATH, {"seed": seed, "best_epoch": run["best_epoch"], "val AUPRC (p = 0)": run["best_val_ap"],
                                       "epoci executate": len(run["history"]), "exemple train": len(y_s), "timp antrenare [s]": run["time_train_s"]})
        train_done[seed] = load_jsonl(TRAIN_RUNS_PATH, required=True)[-1]
        train_time_session += run["time_train_s"]
        source = f"antrenat și salvat ({run['time_train_s']:.0f} s)"
    assert np.isclose(average_precision_score(y_val, predict(Z_val, saved["weights"])), saved["best_val_ap"]), f"seed {seed}: ponderile nu reproduc AUPRC-ul salvat"
    assert len(saved["history"]) == EPOCHS and saved["best_epoch"] == 1 + int(np.argmax([h["validation AUPRC"] for h in saved["history"]]))
    models[seed] = {"weights": saved["weights"], "encoder": encoder, "Z_val": Z_val, "best_epoch": saved["best_epoch"],
                    "history": saved["history"], "initial_weights": saved["initial_weights"]}
    print(f"seed {seed}: {source}   epoca aleasă {saved['best_epoch']:2d}   val AUPRC (p = 0) {saved['best_val_ap']:.4f}")

# baseline-ul (Exp. 2) a pornit din aceleași ponderi inițiale: comparația pe seed este pereche
for seed in SEEDS:
    assert torch.equal(models[seed]["initial_weights"], init_weights(seed).detach()), f"seed {seed}: inițializare diferită"
print("✅ 5 modele Exp. 3: ponderi salvate, AUPRC reprodus; aceleași ponderi inițiale ca baseline-ul (init_weights(seed))")

## 5. Evaluare sub zgomot pe validation

Identic cu etapa 1 din Experimentul 2: pentru fiecare seed și p, scorurile validării (fără zgomot cu `predict`, cu zgomot cu `noisy_predict`), pragul max-F1 pe validation **la același p**, metricile la acest prag și Spearman ρ față de scorurile fără zgomot. Fiecare (seed, p) terminat se salvează imediat în `val_runs.jsonl`.

Precision / recall / F1 / MCC sunt optimiste (prag ales pe același set), exact ca în baseline, deci comparația dintre cele două rămâne corectă.

In [ ]:
w0, Zv0 = models[SEEDS[0]]["weights"], models[SEEDS[0]]["Z_val"]   # sanity: default.mixed la p = 0 = state-vector
sanity = np.abs(noisy_predict(Zv0[:2000], w0, 0.0) - predict(Zv0[:2000], w0)).max()
assert sanity < 1e-6, sanity
print(f"default.mixed la p = 0 vs default.qubit (2000 tranzacții): diferență maximă {sanity:.1e}")

val_done = {(r["seed"], r["p"]) for r in load_jsonl(VAL_RUNS_PATH)}
print(f"treceri zgomotoase de făcut pe validation: {sum((s, p) not in val_done for s in SEEDS for p in NOISE_LEVELS)} din {len(SEEDS) * len(NOISE_LEVELS)}")
for seed in SEEDS:
    m = models[seed]
    clean_val = predict(m["Z_val"], m["weights"])
    for p in P_ALL:
        if (seed, p) in val_done:
            continue
        t0 = time.perf_counter()
        scores = clean_val if p == 0.0 else noisy_predict(m["Z_val"], m["weights"], p)
        elapsed = time.perf_counter() - t0
        threshold = choose_threshold(y_val, scores)
        metrics = evaluate(y_val, scores, threshold)
        record = {"seed": seed, "p": p, **metrics, "Spearman rho": spearmanr(clean_val, scores).statistic,
                  "threshold": float(threshold), "timp scorare [s]": elapsed}
        append_jsonl(VAL_RUNS_PATH, record)
        val_done.add((seed, p))
        print(f"seed {seed}  p = {p:<5}  val AUPRC {metrics['AUPRC']:.3f}  F1 {metrics['F1']:.3f}  rho {record['Spearman rho']:.4f}  prag {threshold:.3f}  ({elapsed:.0f} s)")

## 6. Rezultate Exp. 3 și comparația cu baseline-ul (Exp. 2)

Rezultatele baseline-ului sunt citite din `results/result_exp2/val_runs.jsonl` (doar citit). Toate valorile sunt pe **validation**. Medie ± std (ddof = 1) pe 5 seeds; diferențele sunt **pereche**: Exp. 3 − Exp. 2 pe același seed.

In [ ]:
METRICS = ["AUPRC", "AUROC", "Spearman rho", "precision", "recall", "F1", "MCC"]


def load_results(path):
    records = [r for r in load_jsonl(path, required=True) if r["seed"] in SEEDS and r["p"] in P_ALL]
    keys = [(r["seed"], r["p"]) for r in records]
    missing = sorted({(s, p) for s in SEEDS for p in P_ALL} - set(keys))
    assert not missing, f"lipsesc rezultate (seed, p) în {path}: {missing}"
    assert len(keys) == len(set(keys)), f"rezultate duplicate în {path}"
    return pd.DataFrame(records).sort_values(["p", "seed"]).reset_index(drop=True)


def mean_std(values, digits=3):
    values = np.asarray(values, dtype=float)
    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


new_results = load_results(VAL_RUNS_PATH)
base_results = load_results(EXP2_DIR / "val_runs.jsonl")
detail_cols = ["seed", "p", *METRICS, "threshold", "TP", "FP", "FN", "TN", "timp scorare [s]"]
new_results[detail_cols].to_csv(RESULTS_DIR / "val_detailed.csv", index=False)

summary = pd.DataFrame([
    {"p": p, **{m: mean_std(new_results.loc[new_results["p"] == p, m], 4 if m == "Spearman rho" else 3) for m in METRICS},
     "prag": mean_std(new_results.loc[new_results["p"] == p, "threshold"])}
    for p in P_ALL
]).set_index("p")
summary.to_csv(RESULTS_DIR / "val_summary.csv")
print(f"Exp. 3 (ZZ ×1), medie ± std pe {len(SEEDS)} seeds — validation")
display(summary)


def paired(metric, p):
    base = base_results[base_results["p"] == p].set_index("seed").loc[SEEDS, metric]
    new = new_results[new_results["p"] == p].set_index("seed").loc[SEEDS, metric]
    return base, new, new - base


comparison = []
for p in P_ALL:
    for m in METRICS:
        base, new, diff = paired(m, p)
        digits = 4 if m == "Spearman rho" else 3
        comparison.append({"p": p, "metrică": m, BASE_LABEL: mean_std(base, digits), NEW_LABEL: mean_std(new, digits),
                           "diferență pereche (Exp. 3 − Exp. 2)": f"{diff.mean():+.{digits}f} ± {diff.std(ddof=1):.{digits}f}",
                           "seeds cu Exp. 3 > Exp. 2": f"{int((diff > 0).sum())} / {len(diff)}"})
comparison = pd.DataFrame(comparison).set_index(["p", "metrică"])
comparison.to_csv(RESULTS_DIR / "comparison_vs_exp2.csv")
print(f"Comparație {BASE_LABEL} vs {NEW_LABEL}, medie ± std pe {len(SEEDS)} seeds — validation")
display(comparison)

per_seed = []
for p in P_ALL:
    for seed in SEEDS:
        row = {"p": p, "seed": seed}
        for m, short in [("AUPRC", "AUPRC"), ("Spearman rho", "rho"), ("F1", "F1")]:
            base, new, diff = paired(m, p)
            row.update({f"{short} Exp. 2": base[seed], f"{short} Exp. 3": new[seed], f"Δ {short}": diff[seed]})
        per_seed.append(row)
per_seed = pd.DataFrame(per_seed)
per_seed.to_csv(RESULTS_DIR / "paired_differences.csv", index=False)
print("Diferențe pereche per seed (Exp. 3 − Exp. 2), p = 0.01 și p = 0.05")
display(per_seed[per_seed["p"].isin([0.01, 0.05])].set_index(["p", "seed"]).round(4))

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels = ["0", *[str(p) for p in NOISE_LEVELS]]
suptitle = f"Validation · {BASE_LABEL} vs {NEW_LABEL} · antrenat fără zgomot · medie ± std pe {len(SEEDS)} seeds (gri: seeds)"

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, col, title in zip(axes, ["AUPRC", "F1", "Spearman rho"],
                          ["AUPRC vs p", "F1 la pragul max-F1 (ales la același p) vs p", "Spearman ρ, scoruri cu zgomot vs fără zgomot"]):
    for results, color, marker, label, dx in [(base_results, BASE_COLOR, "o", BASE_LABEL, -0.07), (new_results, NEW_COLOR, "s", NEW_LABEL, 0.07)]:
        groups = [results.loc[results["p"] == p, col].to_numpy(dtype=float) for p in P_ALL]
        for i, g in enumerate(groups):
            ax.scatter(np.full(len(g), xpos[i] + dx), g, s=18, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + dx, [g.mean() for g in groups], yerr=[g.std(ddof=1) for g in groups], color=color, marker=marker,
                    markersize=8, linewidth=2, capsize=4, label=label, zorder=3)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(title)
axes[0].legend(loc="lower left")
fig.suptitle(suptitle, fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "comparison_plots.png", dpi=130)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (col, short, title) in zip(axes, [("AUPRC", "AUPRC", "Δ AUPRC (Exp. 3 − Exp. 2)"), ("Spearman rho", "rho", "Δ Spearman ρ (Exp. 3 − Exp. 2)")]):
    for seed in SEEDS:
        d = per_seed[per_seed["seed"] == seed].set_index("p").loc[P_ALL, f"Δ {short}"]
        ax.plot(xpos, d, color=SEED_DOT_COLOR, marker="o", markersize=5, linewidth=1, alpha=0.8, zorder=2)
        ax.annotate(f"s{seed}", (xpos[-1], d.iloc[-1]), textcoords="offset points", xytext=(6, 0), fontsize=8, color="0.35", va="center")
    mean_d = per_seed.groupby("p")[f"Δ {short}"].mean().loc[P_ALL]
    ax.plot(xpos, mean_d, color=NEW_COLOR, marker="s", markersize=8, linewidth=2, label="medie pe 5 seeds", zorder=3)
    ax.axhline(0, color="0.3", linestyle="--", linewidth=1)
    ax.set_xticks(xpos, xlabels)
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(title)
axes[0].legend(loc="lower left")
fig.suptitle("Diferențe pereche pe seed (gri: câte o linie per seed; > 0 = Exp. 3 mai bun) · validation", fontsize=11)
plt.tight_layout()
fig.savefig(RESULTS_DIR / "paired_differences.png", dpi=130)
plt.show()

## 7. Cost ledger: baseline vs Exp. 3

Patru lucruri diferite, raportate separat:
1. **Structura circuitului** (secțiunea 3): porți native, CNOT, adâncime, locuri de zgomot. Acestea determină cât zgomot acumulează circuitul și cât durează un shot pe hardware.
2. **Execuții și shots pe hardware (nominal)**, cu aceeași formulă ca la baseline: antrenare = epoci × 590 × $(1 + 2P)$, validare după fiecare epocă, trecerea finală pe validation la p = 0 (numărată o dată), câte o trecere pe validation pentru fiecare p cu zgomot, testul (planificat). Numărul de **execuții** depinde de parametri, exemple și epoci, **nu de porți**: cu $P = 32$ în ambele circuite, formula dă același număr. Execuțiile măsurate cu `qml.Tracker` sunt în tabelul de structură.
3. Costul baseline-ului vine din Experimentele 1–2 (citit), al Exp. 3 din propriile rulări.
4. **Timpul de simulare**, separat, informativ.

In [ ]:
n_val = len(val_idx)
exp1_runs = {r["seed"]: r for r in load_jsonl(EXP1_DIR / "runs.jsonl", required=True) if r["ratio"] == 1}
exp2_ledger = pd.read_csv(EXP2_DIR / "cost_ledger.csv", index_col=0)
train_runs = {r["seed"]: r for r in load_jsonl(TRAIN_RUNS_PATH, required=True)}
assert set(SEEDS) <= set(exp1_runs) and set(SEEDS) <= set(train_runs)
test_done = RUN_TEST and TEST_RUNS_PATH.exists()


def ledger_for(epochs_per_seed):
    """(componentă, stare, execuții per seed) cu formula Părții 8; epochs_per_seed = epocile executate per seed."""
    epochs = set(epochs_per_seed)
    assert len(epochs) == 1, "toate seed-urile trebuie să aibă același număr de epoci"
    e = epochs.pop()
    return [
        ("1. antrenare: forward + parameter shift", "executat", e * 590 * EXECUTIONS_PER_SAMPLE),
        ("2. validare după fiecare epocă, p = 0", "executat", e * n_val),
        ("3a. validation p = 0, ponderile alese (numărată o dată)", "executat", n_val),
        ("3b. validation sub zgomot, p = 0.001 / 0.01 / 0.05", "executat", len(NOISE_LEVELS) * n_val),
        ("4. test, p = 0 / 0.001 / 0.01 / 0.05", "executat" if test_done else "planificat (neexecutat)", len(P_ALL) * n_test_rows),
    ]


base_rows = ledger_for([exp1_runs[s]["epoci executate"] for s in SEEDS])
new_rows = ledger_for([train_runs[s]["epoci executate"] for s in SEEDS])
for (name, _, per_seed_exec), (_, row) in zip(base_rows, exp2_ledger.iterrows()):   # baseline-ul coincide cu ledger-ul Exp. 2
    assert per_seed_exec == row["execuții / seed"], f"{name}: {per_seed_exec} vs Exp. 2 {row['execuții / seed']}"

n_s = len(SEEDS)
ledger = pd.DataFrame([
    {"componentă": name, "stare": status,
     f"execuții / seed, {BASE_LABEL}": b, f"execuții / seed, {NEW_LABEL}": n,
     f"execuții {n_s} seeds, {NEW_LABEL}": n * n_s, f"shots {n_s} seeds, {NEW_LABEL}": n * n_s * SHOTS_PER_EXECUTION,
     "diferență execuții (Exp. 3 − Exp. 2)": n - b}
    for (name, status, b), (_, _, n) in zip(base_rows, new_rows)
]).set_index("componentă")
ledger.to_csv(RESULTS_DIR / "cost_ledger.csv")
structure.to_csv(RESULTS_DIR / "circuit_structure.csv")

print("Structura circuitului (determină zgomotul acumulat, nu numărul de execuții):")
display(structure)
print(f"Execuții și shots pe hardware (nominal, parameter shift, {SHOTS_PER_EXECUTION} shots/execuție):")
display(ledger.map(lambda v: f"{v:,}" if isinstance(v, (int, np.integer)) else v))
executed = ledger[ledger["stare"] == "executat"]
print(f"Total executat per seed: baseline {executed[f'execuții / seed, {BASE_LABEL}'].sum():,}   Exp. 3 {executed[f'execuții / seed, {NEW_LABEL}'].sum():,}")
reduction = {k: 1 - structure.loc[NEW_LABEL, k] / structure.loc[BASE_LABEL, k] for k in ["porți native", "CNOT", "adâncime", "locuri de zgomot"]}
print("Reducere structurală față de baseline: " + ", ".join(f"{k} −{v:.0%}" for k, v in reduction.items()))

sim_time = pd.DataFrame([
    {"ce": "antrenare Exp. 3 (5 seeds, din train_runs.jsonl)", "timp simulator [min]": sum(train_runs[s]["timp antrenare [s]"] for s in SEEDS) / 60},
    {"ce": "antrenare baseline 1:1 (Exp. 1, din runs.jsonl)", "timp simulator [min]": sum(exp1_runs[s]["timp antrenare [s]"] for s in SEEDS) / 60},
    {"ce": "treceri zgomotoase pe validation, Exp. 3", "timp simulator [min]": new_results.loc[new_results["p"] > 0, "timp scorare [s]"].sum() / 60},
    {"ce": "treceri zgomotoase pe validation, baseline (Exp. 2)", "timp simulator [min]": base_results.loc[base_results["p"] > 0, "timp scorare [s]"].sum() / 60},
]).set_index("ce")
print("Timp efectiv de simulare (informativ, depinde de mașină; NU este cost hardware):")
display(sim_time.round(1))
sim_time.to_csv(RESULTS_DIR / "simulator_time.csv")

## 8. Evaluarea finală pe test (doar cu `RUN_TEST = True`)

Neexecutată în acest experiment. Când configurația finală este aleasă, `evaluate_under_noise` din notebookul oficial, neschimbat, se rulează pe fiecare seed; pragurile trebuie să coincidă cu cele de pe validation.

In [ ]:
if RUN_TEST:
    val_thresholds = new_results.set_index(["seed", "p"])["threshold"]
    tested = {r["seed"] for r in load_jsonl(TEST_RUNS_PATH)}
    for seed in SEEDS:
        if seed in tested:
            continue
        m = models[seed]
        protocol = evaluate_under_noise(vqc, m["weights"], m["Z_val"], m["encoder"].transform(X_test))
        for p, row in protocol.iterrows():
            assert np.isclose(row["threshold"], val_thresholds[(seed, p)]), f"seed {seed}, p = {p}: pragul diferă de validation"
        for p, row in protocol.iterrows():
            append_jsonl(TEST_RUNS_PATH, {"seed": seed, "p": float(p), **row.to_dict()})
        print(f"seed {seed}: " + "   ".join(f"p = {p}: AUPRC {row['AUPRC']:.3f}" for p, row in protocol.iterrows()))
else:
    print("RUN_TEST = False: testul nu a fost transformat și nu a fost scorat.")

## 9. Cum citim rezultatele

- Întrebarea experimentului: **scade mai puțin** AUPRC-ul și ρ la p = 0.01 și 0.05 cu o singură repetare? Citim atât nivelul absolut (Exp. 3 vs Exp. 2 la același p), cât și **pierderea față de p = 0** a fiecărui model.
- Diferențele sunt pereche (același seed, aceleași date, aceleași ponderi inițiale). O diferență contează doar dacă e mare față de std-ul ei și are același semn la majoritatea seed-urilor. Validation are 99 de fraude.
- Precision / recall / F1 / MCC sunt optimiste pentru ambele modele (prag pe același set); AUPRC, AUROC și ρ nu depind de prag.
- Cost: execuțiile nominale sunt aceleași (aceiași 32 de parametri); câștigul este în porți, adâncime și locuri de zgomot, adică în calitatea fiecărui shot, nu în numărul de shots.
- Concluziile rămân pe validation; raportarea finală cere evaluarea pe test și referința clasică (regresie logistică pe aceleași unghiuri).